# TP2 : Mémoire de conversation

**Contexte :** assistant du service client de Lagune Banque.

Le client transmet les informations de son remboursement en plusieurs messages. Vous allez ajouter une mémoire à l'agent, vérifier ce qu'il retient et comparer le coût de l'historique complet, de sa troncature et de son résumé.

Complétez les `# TODO`. Références : [mémoire à court terme](https://docs.langchain.com/oss/python/langchain/short-term-memory), [agents](https://docs.langchain.com/oss/python/langchain/agents).

## <a id="configuration"></a>Configuration

In [1]:
import sys
from pathlib import Path

RACINE = next(p for p in [Path("."), Path(".."), Path("../..")] if (p / "code_partage").exists()).resolve()
sys.path[:0] = [str(RACINE / "code_partage"), str(RACINE / "03_agents" / "socle")]

import re
import time

import pandas as pd
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, wrap_model_call
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

import tools
from base import AUJOURDHUI, Context, new_model, reset_db
from config import prepare

In [2]:
prepare()

Connexion OK : gemini/gemini-3.5-flash-lite


## <a id="syntaxe"></a>Syntaxe

Ce TP utilise trois éléments de LangChain et de LangGraph, que vous allez écrire :

```python
# 1. Mémoire : un checkpointer sauvegarde l'état de l'agent (ses messages) sous un thread_id
agent = create_agent(model, tools, checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "client-1"}}        # un thread_id = une conversation
agent.invoke({"messages": [{"role": "user", "content": "Bonjour"}]}, config)

# 2. Middleware : du code exécuté autour de chaque appel au modèle
@wrap_model_call
def window(request, handler):
    return handler(request.override(messages=request.messages[-6:]))   # ce que le modèle reçoit

# 3. Middleware fourni : résumé des anciens messages
create_agent(model, tools, middleware=[SummarizationMiddleware(model=model, trigger=("messages", 12), keep=("messages", 6))])
```

L'historique est une liste de messages : `HumanMessage` (le client), `AIMessage` (le modèle ; son attribut `tool_calls` liste les tools demandés) et `ToolMessage` (le résultat d'un tool).

## <a id="scenario"></a>Scénario

Le client est M. Kouassi (CLI-0002). L'agent doit recueillir trois informations, une par message, puis vérifier la transaction. Après la vérification, le client pose quatre questions de suivi, dont une appelle une recherche dans la documentation : l'historique grossit à chaque tour. Le dernier message est la **question de contrôle** : elle demande de rappeler les trois informations.

In [3]:
TOOLS = [tools.search_docs, tools.get_customer_loans, tools.get_schedule, tools.check_lm_transaction]

PROMPT = f"""Tu es l'assistant automatisé du service client de Lagune Banque (un programme, pas une personne). Nous sommes le {AUJOURDHUI} ; « hier » est la veille.
Tu réponds en français, en phrases courtes. Un client signale un remboursement non pris en compte.
Recueille, un message après l'autre et dans cet ordre : la référence du prêt, la date du remboursement, le montant. Ne devine rien et n'appelle aucun tool avant d'avoir les trois.
Quand tu les as, vérifie avec check_lm_transaction, puis réponds. Ne redemande jamais une information déjà donnée."""

MESSAGES = [
    "Bonjour, j'ai un problème avec un remboursement.",
    "Mon prêt est PRT-0002.",
    "J'ai remboursé hier, le 15 novembre.",
    "31 500 FCFA.",
    "Et quel est le statut de ma transaction ?",
    "Y a-t-il des frais de retard sur cette échéance ?",
    "Quel est le solde restant de mon prêt ?",
    "Quel est le délai de traitement d'une réclamation ?",
    "Rappelle-moi en une phrase la référence du prêt, la date du remboursement et le montant que je t'ai donnés.",
]

## <a id="sans-memoire"></a>Sans mémoire

`converse` envoie les messages un par un, recueille les réponses et mesure les tokens d'entrée de chaque appel au modèle.

Sans `thread_id`, chaque message est envoyé seul : l'agent ne voit pas les précédents.

In [ ]:
def build_agent(**options):
    return create_agent(new_model(), TOOLS, system_prompt=PROMPT, context_schema=Context, **options)


def last_turn(result):
    messages = result["messages"]
    start = max(i for i, m in enumerate(messages) if isinstance(m, HumanMessage))
    return messages[start:]


class InputTokens(BaseCallbackHandler):
    def __init__(self):
        self.calls = []

    def on_llm_end(self, response, **kwargs):
        self.calls.append(response.generations[0][0].message.usage_metadata["input_tokens"])


def converse(agent, thread_id=None, messages=MESSAGES):
    reset_db()
    counter = InputTokens()
    config = {"callbacks": [counter], **({"configurable": {"thread_id": thread_id}} if thread_id else {})}
    replies, first_calls = [], []
    for text in messages:
        result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config, context=Context(client_id="CLI-0002"))
        turn = last_turn(result)
        replies.append(str(turn[-1].content))
        first_calls.append(next(m for m in turn if isinstance(m, AIMessage)).usage_metadata["input_tokens"])
    return {"replies": replies, "first_calls": first_calls, "calls": len(counter.calls), "input_tokens": sum(counter.calls)}


agent_sans_memoire = build_agent()
sans_memoire = converse(agent_sans_memoire)
for question, reponse in list(zip(MESSAGES, sans_memoire["replies"]))[-2:]:
    print(f"CLIENT    : {question}\nASSISTANT : {reponse}\n")

CLIENT    : Quel est le délai de traitement d'une réclamation ?
ASSISTANT : Le délai de traitement d'une réclamation est de 72 heures (DOC-08, DOC-09).

CLIENT    : Rappelle-moi en une phrase la référence du prêt, la date du remboursement et le montant que je t'ai donnés.
ASSISTANT : Vous ne m'avez pas encore communiqué ces informations. 

Quelle est la référence de votre prêt ?



## <a id="memoire-de-lagent"></a>Mémoire de l'agent

Un **checkpointer** sauvegarde l'état de l'agent, dont ses messages, sous un identifiant de fil (`thread_id`). À chaque message du même fil, l'historique complet est renvoyé au modèle.

**TODO 1** : créez `agent_memoire` avec `build_agent`, en lui passant `checkpointer=InMemorySaver()`.

In [5]:
agent_memoire = build_agent(checkpointer=InMemorySaver())   # TODO 1 : ajoutez checkpointer=InMemorySaver()


## <a id="mesure-de-la-memoire"></a>Mesure de la mémoire

« L'agent a retenu » ne se juge pas à l'œil. La question de contrôle exige les trois informations : on cherche chacune dans la réponse.

`missing_elements(reponse)` est fournie : elle renvoie la liste des éléments absents parmi `"prêt"`, `"date"`, `"montant"`, avec une expression régulière par élément.

In [6]:
def missing_elements(reponse):
    motifs = {"prêt": r"PRT-0002", "date": r"hier|15 novembre|2026-11-15|15/11", "montant": r"31[\s  .]?500"}
    return [nom for nom, motif in motifs.items() if not re.search(motif, reponse, re.IGNORECASE)]

In [7]:
print(missing_elements("Prêt PRT-0002, remboursé hier, 31 500 FCFA."))
print(missing_elements("Vous n'avez pas encore indiqué la référence du prêt."))

[]
['prêt', 'date', 'montant']


Le même critère s'applique aux deux agents :

In [10]:
avec_memoire = converse(agent_memoire, "kouassi-1")
print(avec_memoire["replies"][-1])
print("Éléments manquants, sans mémoire :", missing_elements(sans_memoire["replies"][-1]))
print("Éléments manquants, avec mémoire :", missing_elements(avec_memoire["replies"][-1]))

Vous m'avez transmis la référence de prêt PRT-0002, la date de remboursement du 15 novembre et le montant de 31 500 FCFA.
Éléments manquants, sans mémoire : ['prêt', 'date', 'montant']
Éléments manquants, avec mémoire : []


## <a id="isolation-des-conversations"></a>Isolation des conversations

L'état est enregistré par `thread_id`. Avec un autre identifiant, l'agent repart de zéro : c'est ce qui sépare les clients. Le `thread_id` doit venir de la session identifiée.

**TODO 2** : posez la question de contrôle (`MESSAGES[-1]`) à `agent_memoire` sur un nouveau fil, `"kouassi-2"`. Appelez `agent_memoire.invoke(...)` avec le message au format `{"messages": [{"role": "user", "content": ...}]}`, la config `{"configurable": {"thread_id": "kouassi-2"}}` et `context=Context(client_id="CLI-0002")`.

In [14]:
# TODO 2 : invoquez agent_memoire (question de contrôle, thread_id "kouassi-2", context=Context(client_id="CLI-0002"))
autre_fil = agent_memoire.invoke({"messages": [{"role": "user", "content": MESSAGES[-1]}]}, {"configurable": {"thread_id": "kouassi-2"}}, context=Context(client_id="CLI-0002"))

if autre_fil is not None:
    reponse = str(autre_fil["messages"][-1].content)
    print(reponse)
    print("Éléments manquants, autre fil :", missing_elements(reponse))
    for fil in ("kouassi-1", "kouassi-2"):
        messages = agent_memoire.get_state({"configurable": {"thread_id": fil}}).values["messages"]
        print(f"Fil {fil} : {len(messages)} messages sauvegardés")

Vous m'avez fourni la référence de prêt « Qultesolderestantdepronet », la date de remboursement « u » et le montant « e ». 

Quelle est la référence exacte de votre prêt ?
Éléments manquants, autre fil : ['prêt', 'date', 'montant']
Fil kouassi-1 : 50 messages sauvegardés
Fil kouassi-2 : 82 messages sauvegardés


## <a id="tronquer-lhistorique"></a>Tronquer l'historique

L'historique grossit à chaque tour : coût et latence augmentent, jusqu'à dépasser la fenêtre de contexte. Première option : ne garder que les derniers **tours** (un tour = un message du client et tout ce qui suit, appels de tools compris). Couper au message près séparerait un appel de tool de son résultat.

`keep_last_turns(messages, k)` est fournie : elle renvoie les messages à partir du `k`-ième dernier `HumanMessage`, ou tous les messages s'il y a moins de `k` tours.

**TODO 3** : écrivez le middleware `window`, décoré par `@wrap_model_call`. Il appelle `handler` avec la requête dont les messages sont remplacés par `keep_last_turns(request.messages, TOURS_GARDES)` (`request.override(messages=...)`). Il s'applique à ce qui est envoyé au modèle ; l'état sauvegardé reste complet.

In [17]:
def keep_last_turns(messages, k):
    starts = [i for i, m in enumerate(messages) if isinstance(m, HumanMessage)]
    return messages[starts[-k]:] if len(starts) > k else messages


TOURS_GARDES = 3

# TODO 3 : middleware `window`, décoré par @wrap_model_call, qui renvoie handler(request.override(messages=...))
@wrap_model_call
def window(request, handler):
    return handler(request.override(messages=keep_last_turns(request.messages, TOURS_GARDES))) 


agent_troncature = build_agent(checkpointer=InMemorySaver(), middleware=[window] if window else [])
troncature = converse(agent_troncature, "kouassi-3")
print(troncature["replies"][-1])
print("Éléments manquants, troncature :", missing_elements(troncature["replies"][-1]))

Vous n'avez pas encore fourni ces informations. Pourriez-vous me donner la référence du prêt concerné ?
Éléments manquants, troncature : ['prêt', 'date', 'montant']


## <a id="bonus-resumer-lhistorique"></a>Bonus : résumer l'historique

Deuxième option : remplacer les anciens messages par un résumé produit par le modèle, et garder les derniers. Un détail est conservé s'il figure dans le résumé, au prix d'un appel au modèle de plus.

**TODO 4 (bonus)** : créez `agent_resume` avec `SummarizationMiddleware(model=new_model(), trigger=("messages", 12), keep=("messages", 6))`, en plus du checkpointer. Ce TODO n'est pas contrôlé.

In [18]:
#agent_resume = None   # TODO 4 (bonus)
agent_resume = build_agent(checkpointer=InMemorySaver(), 
                           middleware=[SummarizationMiddleware(model=new_model(), trigger=("messages", 12), keep=("messages", 6))])

if agent_resume is not None:
    resume = converse(agent_resume, "kouassi-4")
    print(resume["replies"][-1])
    print("Éléments manquants, résumé :", missing_elements(resume["replies"][-1]))

Vous m'avez communiqué la référence de prêt PRT-0002, la date du remboursement du 15 novembre 2026 et le montant de 31 500 FCFA.
Éléments manquants, résumé : []


## <a id="comparaison-des-strategies"></a>Comparaison des stratégies

Les appels au modèle et les tokens d'entrée comptent tous les appels, résumé compris. La trajectoire de l'agent varie d'une exécution à l'autre : comparez les tokens par appel plutôt que les totaux. Le détail du tour 1 est conservé si aucun élément ne manque dans la réponse à la question de contrôle.

In [19]:
VARIANTES = [("sans mémoire", sans_memoire), ("avec mémoire", avec_memoire), (f"troncature, {TOURS_GARDES} derniers tours", troncature)]
if agent_resume is not None:
    VARIANTES.append(("résumé", resume))

MESURE = pd.DataFrame([{"variante": nom, "appels au modèle": r["calls"], "tokens d'entrée": r["input_tokens"],
                        "tokens par appel": round(r["input_tokens"] / r["calls"]),
                        "éléments manquants": len(missing_elements(r["replies"][-1]))} for nom, r in VARIANTES]).set_index("variante")
MESURE

,appels au modèle,tokens d'entrée,tokens par appel,éléments manquants
variante,,,,
sans mémoire,10,6728,673,3
avec mémoire,12,32824,2735,0
"troncature, 3 derniers tours",14,15492,1107,3
résumé,18,24899,1383,0


Tokens du premier appel au modèle de chaque tour : c'est la taille de l'historique envoyé.

In [20]:
PAR_TOUR = pd.DataFrame({nom: r["first_calls"] for nom, r in VARIANTES}, index=range(1, len(MESSAGES) + 1))
PAR_TOUR.index.name = "tour"
PAR_TOUR

,sans mémoire,avec mémoire,"troncature, 3 derniers tours",résumé
tour,,,,
1,617,2188,617,617
2,617,2261,658,651
3,620,2349,684,677
4,615,2420,662,699
5,615,2475,1047,1718
6,621,2524,1065,1744
7,616,2766,1408,1803
8,618,3013,1414,1358
9,633,3605,1253,1951


## <a id="sur-le-chat"></a>Sur le chat

L'agent avec mémoire sur l'interface Gradio. Le fil est celui de la session : les messages suivants s'appuient sur les précédents.

In [ ]:
import chat
import front

demo = chat.create(agent_memoire, thread=True, tag="m3-memoire", name="m3-tp2-memoire")
front.launch(demo)

In [ ]:
demo.close()

## Contrôle

Cette cellule affiche **ÉCHEC** tant que les `# TODO` ne sont pas faits.

In [21]:
try:
    assert agent_memoire.checkpointer is not None, "agent_memoire doit avoir un checkpointer (TODO 1)"
    assert MESURE.loc["avec mémoire", "éléments manquants"] == 0, "avec mémoire, l'agent doit retrouver les trois éléments (TODO 1)"
    assert autre_fil is not None, "invoquez agent_memoire sur le fil kouassi-2 (TODO 2)"
    fil_2 = agent_memoire.get_state({"configurable": {"thread_id": "kouassi-2"}}).values["messages"]
    assert sum(isinstance(m, HumanMessage) for m in fil_2) == 1, "le fil kouassi-2 ne doit contenir que la question de contrôle (TODO 2)"
    assert window is not None, "écrivez le middleware window (TODO 3)"
    exemple = [HumanMessage("a"), AIMessage("b"), HumanMessage("c"), AIMessage("d"), HumanMessage("e"), AIMessage("f")]
    assert [m.content for m in keep_last_turns(exemple, 2)] == ["c", "d", "e", "f"], "keep_last_turns doit garder les 2 derniers tours"
    assert PAR_TOUR[f"troncature, {TOURS_GARDES} derniers tours"].iloc[-1] < PAR_TOUR["avec mémoire"].iloc[-1], "la troncature doit envoyer moins de tokens au dernier tour (TODO 3)"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")

ÉCHEC : le fil kouassi-2 ne doit contenir que la question de contrôle (TODO 2)
